In [1]:
import sys, os
from pathlib import Path

# Make `from src...` imports and the relative `outputs/` paths work regardless
# of which directory the Jupyter kernel started in (VS Code defaults the kernel
# cwd to the workspace root, not this project folder). We locate the project
# root — the directory containing `src/config.py` — then chdir into it.
def _find_project_root(start: Path) -> Path:
    for base in [start, *start.parents]:
        if (base / "src" / "config.py").exists():       # root is an ancestor
            return base
        hits = list(base.glob("*/src/config.py"))        # ...or one level down
        if hits:
            return hits[0].parent.parent
    raise FileNotFoundError("Could not find project root containing src/config.py")

_root = _find_project_root(Path.cwd())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Project root:", _root)

import pandas as pd
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import json

from src.config import PROCESSED_DATA_DIR, POSTERIOR_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.model_pymc import build_and_fit, load_trace
from src.predict import predict_all, predict_match

Project root: c:\Users\Jonathan\OneDrive\Codes\soccer-predict\CS179-Match-Prediction-Project


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [2]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates=["Date"])
team_mapping = json.load(open(PROCESSED_DATA_DIR / "team_mapping.json"))

train_df = df[df["season"].isin(TRAIN_SEASONS)].reset_index(drop=True)
test_df  = df[df["season"].isin(TEST_SEASONS)].reset_index(drop=True)

print(f"Training on {len(train_df)} matches, {len(team_mapping)} teams")
print(f"Testing on  {len(test_df)} matches")

Training on 2660 matches, 31 teams
Testing on  380 matches


In [ ]:
# Fit the model — takes ~10-20 min on CPU for 7 seasons
# To skip fitting and load a previously saved trace instead, comment out
# the line below and uncomment the one after it.
idata = build_and_fit(train_df, team_mapping, draws=1000, tune=500)
# idata = load_trace("trace.nc")

In [ ]:
# Summarise the global parameters and check convergence.
# r_hat measures whether the MCMC chains agree with each other —
# values close to 1.0 mean the chains converged; >= 1.01 is a warning sign.
summary = az.summary(idata, var_names=["mu", "home_adv", "sigma_att", "sigma_def"])
display(summary)

assert (summary["r_hat"] < 1.01).all(), "Convergence warning: r_hat >= 1.01 for some parameters"

In [ ]:
# Trace plots: left column shows the full chain (should look like fuzzy caterpillars),
# right column shows the marginal distribution of each parameter.
az.plot_trace(idata, var_names=["mu", "home_adv", "sigma_att", "sigma_def"])
plt.tight_layout()
plt.show()

In [ ]:
preds = predict_all(test_df, idata)
display(
    preds[["HomeTeam", "AwayTeam", "FTR", "p_home_win", "p_draw", "p_away_win", "mode_scoreline"]]
    .head(10)
)

In [ ]:
# Predict a single match by name — useful for interactive exploration.
result = predict_match("Arsenal", "Chelsea", idata, team_mapping)
print(result)

In [ ]:
import os
os.makedirs("outputs/tables", exist_ok=True)
preds.to_csv("outputs/tables/bayesian_preds.csv", index=False)
print("Saved outputs/tables/bayesian_preds.csv")

In [ ]:
# Fit both variants and save their predictions.
# Compare against bayesian_preds.csv in the results notebook to see
# what each simplification costs in predictive accuracy.
from src.model_pymc import build_and_fit_no_home_adv, build_and_fit_single_strength

idata_nha = build_and_fit_no_home_adv(train_df, team_mapping)
idata_ss  = build_and_fit_single_strength(train_df, team_mapping)

preds_nha = predict_all(test_df, idata_nha)
preds_ss  = predict_all(test_df, idata_ss)

preds_nha.to_csv("outputs/tables/bayesian_preds_no_home_adv.csv",    index=False)
preds_ss.to_csv( "outputs/tables/bayesian_preds_single_strength.csv", index=False)
print("Saved variation predictions.")

## Evaluation metrics beyond accuracy

Accuracy only checks whether the most likely outcome was correct and discards the probabilities entirely. The cells below score the **full predicted distribution**:

- **Proper scoring rules** (log loss, RPS, Brier) — reward sharp *and* calibrated probabilities.
- **Reliability diagram + ECE** — are the probabilities calibrated?
- **Goal-level errors** — MAE/RMSE on expected goals and exact-scoreline accuracy.

Everything reads from `outputs/tables/bayesian_preds.csv`, so re-run the fitting cells above first (or just the save cell) if that file doesn't exist yet.

In [3]:
# --- Baseline metrics only (NO model fit required) ---
# The true outcomes come from the data and the baselines don't use the model,
# so this runs in seconds. Run cells 1-2 (imports + data load) first, then this
# cell — you can stop here while the MCMC fit waits. These rows are also the
# reference the fitted model has to beat.
from sklearn.metrics import log_loss

ORDER = ["H", "D", "A"]                       # ordinal scale: home win -> draw -> away win

y = test_df["FTR"].to_numpy()                 # actual test outcomes (no model needed)
Y = np.stack([(y == c) for c in ORDER], axis=1).astype(float)

def _rps(probs, onehot):
    cp, co = np.cumsum(probs, axis=1), np.cumsum(onehot, axis=1)
    return np.mean(((cp - co) ** 2).sum(axis=1) / (probs.shape[1] - 1))

def _score(probs):
    return {"Log loss": log_loss(y, probs, labels=ORDER),
            "RPS":       _rps(probs, Y),
            "Brier":     np.mean(((probs - Y) ** 2).sum(axis=1))}

N = len(y)
# Uniform: always 1/3 each. Class-prior: the H/D/A rates in the *training* data
# (an honest baseline — no peeking at the test labels).
train_y = train_df["FTR"].to_numpy()
priors  = np.stack([(train_y == c) for c in ORDER], axis=1).mean(axis=0)

baselines = pd.DataFrame({
    "Uniform (1/3 each)":        _score(np.full((N, 3), 1 / 3)),
    "Training base-rates":       _score(np.tile(priors, (N, 1))),
}).T
display(baselines.round(4))
print("Training H/D/A base-rates:", dict(zip(ORDER, priors.round(3))))

c:\Users\Jonathan\OneDrive\Codes\soccer-predict\.venv\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['H', 'D', 'A']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['A', 'D', 'H'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(
c:\Users\Jonathan\OneDrive\Codes\soccer-predict\.venv\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['H', 'D', 'A']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['A', 'D', 'H'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(


,Log loss,RPS,Brier
Uniform (1/3 each),1.0986,0.2418,0.6667
Training base-rates,1.0973,0.2338,0.6373


Training H/D/A base-rates: {'H': np.float64(0.442), 'D': np.float64(0.235), 'A': np.float64(0.323)}


In [4]:
# --- Evaluation metrics setup ---
# These cells read the saved predictions so they can run on their own. If you
# just ran the cells above, `preds` is already in memory and you could pass
# that to the functions below instead of re-reading the CSV.
from sklearn.metrics import log_loss

ORDER     = ["H", "D", "A"]                       # ordinal scale: home win -> draw -> away win
PROB_COLS = ["p_home_win", "p_draw", "p_away_win"]

eval_df = pd.read_csv("outputs/tables/bayesian_preds.csv")
P = eval_df[PROB_COLS].to_numpy()                                  # (N, 3) predicted probs
y = eval_df["FTR"].to_numpy()                                      # (N,)   actual outcomes
Y = np.stack([(y == c) for c in ORDER], axis=1).astype(float)     # (N, 3) one-hot actuals

print(f"Scoring {len(eval_df)} test matches "
      f"({(y == 'H').mean():.0%} H / {(y == 'D').mean():.0%} D / {(y == 'A').mean():.0%} A)")

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/tables/bayesian_preds.csv'

### Proper scoring rules

These reward sharp **and** honest probabilities, unlike accuracy which only checks the top pick. **RPS** is the standard in football forecasting because it respects the ordinal H–D–A scale; **log loss** is the held-out log-likelihood (natural for a Bayesian model); **Brier** is the multiclass mean squared error of the probability vector. Lower is better for all three. The uniform `1/3, 1/3, 1/3` row is a reference point — the model should beat it.

In [ ]:
def ranked_probability_score(probs, onehot):
    """Mean Ranked Probability Score over matches. Order-aware: predicting a
    draw when the result was an away win is penalised less than predicting a
    home win, because the H-D-A scale is ordinal. Columns must be in ordinal
    order. Lower is better."""
    cum_pred = np.cumsum(probs, axis=1)
    cum_obs  = np.cumsum(onehot, axis=1)
    r = probs.shape[1]
    return np.mean(((cum_pred - cum_obs) ** 2).sum(axis=1) / (r - 1))

def proper_scores(probs):
    return {
        "Log loss": log_loss(y, probs, labels=ORDER),
        "RPS":      ranked_probability_score(probs, Y),
        "Brier":    np.mean(((probs - Y) ** 2).sum(axis=1)),
    }

pred_class = np.array(ORDER)[P.argmax(axis=1)]
accuracy   = (pred_class == y).mean()

# Reference baseline: always predict 1/3, 1/3, 1/3
P_uniform = np.full_like(P, 1 / 3)

table = pd.DataFrame({
    "Bayesian model":   proper_scores(P),
    "Uniform baseline": proper_scores(P_uniform),
}).T
table.insert(0, "Accuracy", [accuracy, np.nan])
display(table.round(4))

### Calibration

Are the probabilities *honest*? A reliability diagram bins predictions by confidence and checks whether, e.g., matches predicted at 60% actually happen ~60% of the time. **ECE** (Expected Calibration Error) is the single-number summary — the average gap between confidence and accuracy. Lower is better.

In [ ]:
def reliability(probs, y_true, classes, n_bins=10):
    """Confidence-based reliability: bin by the top predicted probability and
    compare the average confidence in each bin to the observed accuracy.
    Returns the Expected Calibration Error and the per-bin points."""
    conf    = probs.max(axis=1)
    pred    = np.array(classes)[probs.argmax(axis=1)]
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    idx  = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)

    ece, xs, ys, ns = 0.0, [], [], []
    for b in range(n_bins):
        m = idx == b
        if not m.any():
            continue
        acc, c = correct[m].mean(), conf[m].mean()
        ece += m.sum() / len(conf) * abs(acc - c)
        xs.append(c); ys.append(acc); ns.append(m.sum())
    return ece, np.array(xs), np.array(ys), np.array(ns)

ece, conf_x, acc_y, counts = reliability(P, y, ORDER)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
ax.scatter(conf_x, acc_y, s=counts * 4, zorder=3, label="model (size = #matches)")
ax.set_xlabel("Predicted confidence (top class)")
ax.set_ylabel("Observed accuracy")
ax.set_title(f"Reliability diagram  (ECE = {ece:.3f})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout(); plt.show()

print(f"Expected Calibration Error (ECE): {ece:.4f}")

### Goal-level metrics

The model is generative over goals, so we can also score the expected goals (`exp_home_goals`, `exp_away_goals`) against the actual scoreline with MAE/RMSE, and check how often the single most-likely scoreline was exactly right.

In [ ]:
# Goal-level metrics — the model also predicts expected goals and a most-likely scoreline.
err_home = eval_df["exp_home_goals"] - eval_df["FTHG"]
err_away = eval_df["exp_away_goals"] - eval_df["FTAG"]

actual_scoreline = eval_df["FTHG"].astype(int).astype(str) + "-" + eval_df["FTAG"].astype(int).astype(str)
exact_scoreline_acc = (eval_df["mode_scoreline"] == actual_scoreline).mean()

display(pd.Series({
    "MAE home goals":           err_home.abs().mean(),
    "MAE away goals":           err_away.abs().mean(),
    "RMSE home goals":          np.sqrt((err_home ** 2).mean()),
    "RMSE away goals":          np.sqrt((err_away ** 2).mean()),
    "Exact scoreline accuracy": exact_scoreline_acc,
}).round(4).to_frame("value"))

### Model comparison with LOO / WAIC (optional)

The cells above score the *out-of-sample* test predictions. To rank the three model variants the Bayesian way, use leave-one-out cross-validation (`az.loo`) or WAIC via `az.compare`. These need the pointwise **log-likelihood**, which the current traces don't store — `build_and_fit` calls `pm.sample(...)` without it.

To enable it, add `idata_kwargs={"log_likelihood": True}` to the `pm.sample(...)` calls in `src/model_pymc.py`, re-fit, then:

```python
az.compare({
    "full":            idata,
    "no_home_adv":     idata_nha,
    "single_strength": idata_ss,
})
```

`az.compare` ranks models by expected log predictive density and is the in-sample complement to the held-out scores above.